In [ ]:
!pip install feedparser spacy sumy nltk pandas
!pip install kagglehub spacy rdflib networkx matplotlib pandas feedparser
!python -m spacy download en_core_web_sm
!pip install beautifulsoup4 lxml


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import nltk

# Télécharger le tokenizer nécessaire pour Sumy
nltk.download('punkt')       # le tokenizer classique
nltk.download('punkt_tab')   # parfois Sumy réclame aussi 'punkt_ta

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True


*********************************************************************************************************************
**AJOUT ANTOLOGIE**

***********************************************************************************************************************************************************


In [ ]:
import feedparser
import spacy
import hashlib
from rdflib import Graph, Namespace, URIRef, Literal, RDF
from rdflib.namespace import XSD, FOAF
import networkx as nx
import matplotlib.pyplot as plt
import nltk

# ===== Sumy =====
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

nltk.download('punkt')

# ==============================
# 1. Configuration
# ==============================
RSS_URL = "https://towardsdatascience.com/feed"
SUMMARY_SENTENCES = 3  # nombre de phrases pour le résumé

# ==============================
# 2. Chargement du flux RSS
# ==============================
feed = feedparser.parse(RSS_URL)
print(f"{len(feed.entries)} articles détectés dans le flux RSS\n")

# ==============================
# 3. Chargement spaCy
# ==============================
nlp = spacy.load("en_core_web_sm")

# ==============================
# 4. Namespaces RDF
# ==============================
EX = Namespace("http://example.org/")
SCHEMA = Namespace("http://schema.org/")

# ==============================
# 5. Définition de la source (publisher)
# ==============================
source_name = feed.feed.get("title", "Unknown Source")
source_url = feed.feed.get("link", RSS_URL)
source_hash = hashlib.md5(source_name.encode("utf-8")).hexdigest()
source_uri = EX[f"source/{source_hash}"]

# ==============================
# 6. Anthologie globale
# ==============================
global_keywords = {}

# ==============================
# 7. Traitement des articles RSS
# ==============================
for idx, entry in enumerate(feed.entries, start=1):
    title = entry.get("title", "").strip()
    summary = entry.get("summary", "").strip()
    content = ""
    if "content" in entry:
        content = " ".join(c.get("value", "") for c in entry.content)
    article_text = "\n".join([title, summary, content]).strip()
    if not article_text:
        continue

    article_url = entry.get("link", f"rss://article/{idx}")
    article_hash = hashlib.md5(article_url.encode("utf-8")).hexdigest()
    article_uri = EX[f"article/{article_hash}"]

    published = entry.get("published", entry.get("updated", None))
    published_display = published if published else "N/A"

    # ==============================
    # 8. Génération du résumé automatique avec Sumy
    # ==============================
    parser = PlaintextParser.from_string(article_text, Tokenizer("english"))
    summarizer = LexRankSummarizer()
    summary_sentences = summarizer(parser.document, SUMMARY_SENTENCES)
    summary_text = " ".join([str(sentence) for sentence in summary_sentences])

    # ==============================
    # 9. Extraction des mots-clés avec spaCy
    # ==============================
    doc = nlp(article_text)
    keywords = set()

    for ent in doc.ents:  # entités nommées
        if ent.label_ in ["ORG", "PERSON", "GPE", "PRODUCT", "EVENT"]:
            keywords.add(ent.text.strip())

    for token in doc:  # noms communs et propres importants
        if token.pos_ in ["NOUN", "PROPN"] and len(token.text) > 2:
            keywords.add(token.text.strip())

    # mettre à jour l’anthologie globale
    for kw in keywords:
        global_keywords[kw] = global_keywords.get(kw, 0) + 1

    # ==============================
    # 10. Impression résumé article + mots-clés
    # ==============================
    print(f"\n=== ARTICLE {idx} ===")
    print(f"Titre : {title}")
    print(f"URL   : {article_url}")
    print(f"Date  : {published_display}")
    print(f"Résumé automatique : {summary_text}")
    print(f"Mots-clés : {', '.join(list(keywords)[:10])}")  # top 10 pour l'affichage

    # ==============================
    # 11. Création du graphe RDF spécifique à cet article
    # ==============================
    g = Graph()
    g.bind("ex", EX)
    g.bind("schema", SCHEMA)
    g.bind("foaf", FOAF)

    # Source
    g.add((source_uri, RDF.type, SCHEMA.Organization))
    g.add((source_uri, SCHEMA.name, Literal(source_name, datatype=XSD.string)))
    g.add((source_uri, SCHEMA.url, Literal(source_url, datatype=XSD.anyURI)))

    # Article
    g.add((article_uri, RDF.type, SCHEMA.Article))
    g.add((article_uri, SCHEMA.headline, Literal(title, datatype=XSD.string)))
    g.add((article_uri, SCHEMA.url, Literal(article_url, datatype=XSD.anyURI)))
    g.add((article_uri, SCHEMA.publisher, source_uri))
    if published:
        g.add((article_uri, SCHEMA.datePublished, Literal(published, datatype=XSD.string)))
    g.add((article_uri, SCHEMA.abstract, Literal(summary_text, datatype=XSD.string)))
    g.add((article_uri, SCHEMA.keywords, Literal(", ".join(keywords), datatype=XSD.string)))

    # Organisations extraites
  # Organisations extraites uniquement depuis les entités ORG de spaCy
orgs = [ent.text.strip() for ent in doc.ents if ent.label_ == "ORG"]

for org in orgs:
    org_hash = hashlib.md5(org.lower().encode("utf-8")).hexdigest()
    org_uri = EX[f"org/{org_hash}"]
    g.add((org_uri, RDF.type, SCHEMA.Organization))
    g.add((org_uri, SCHEMA.name, Literal(org, datatype=XSD.string)))
    g.add((article_uri, SCHEMA.about, org_uri))

    # ==============================
    # 12. Visualisation du graphe RDF pour cet article
    # ==============================
    nx_graph = nx.DiGraph()
    for s, p, o in g:
        nx_graph.add_node(s, label=str(s).split('/')[-1])
        nx_graph.add_node(o, label=str(o).split('/')[-1])
        nx_graph.add_edge(s, o, label=str(p).split('/')[-1])

    pos = nx.spring_layout(nx_graph, k=0.5, iterations=50)
    plt.figure(figsize=(10, 6))
    nx.draw(
        nx_graph, pos,
        with_labels=True,
        labels=nx.get_node_attributes(nx_graph, 'label'),
        node_size=2000,
        node_color='lightblue',
        font_size=9,
        font_weight='bold',
        arrowsize=20
    )
    edge_labels = nx.get_edge_attributes(nx_graph, 'label')
    nx.draw_networkx_edge_labels(nx_graph, pos, edge_labels=edge_labels, font_size=7)
    plt.title(f"Graphe RDF : {title[:50]}...", fontsize=14)
    plt.axis('off')
    plt.show()

# ==============================
# 13. Affichage de l’anthologie globale
# ==============================
print("\n=== ANTHOLOGIE GLOBALE DES MOTS-CLÉS ===")
sorted_keywords = sorted(global_keywords.items(), key=lambda x: x[1], reverse=True)
for kw, freq in sorted_keywords[:20]:
    print(f"{kw} ({freq})")


ModuleNotFoundError: No module named 'rdflib'


*********************************************************************************************************************
**AJOUT PLUSIEURS FLUX RSS**
***********************************************************************************************************************************************************

In [ ]:
!pip install sumy
!pip install feedparser
!pip install beautifulsoup4
import feedparser
import spacy
import hashlib
from rdflib import Graph, Namespace, Literal, RDF
from rdflib.namespace import XSD, FOAF
import networkx as nx
import matplotlib.pyplot as plt
import nltk

# ===== Sumy =====
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

# NLTK tokenizer
nltk.download('punkt')

# ==============================
# 1. Configuration
# ==============================

RSS_FEEDS = [
    "https://towardsdatascience.com/feed",
    "https://www.datasciencecentral.com/feed/",
    "https://www.kdnuggets.com/feed",
    "https://aws.amazon.com/blogs/machine-learning/feed/",
    "https://blogs.nvidia.com/feed/",
    "https://www.wired.com/feed/tag/ai/latest/rss",
    "http://machinelearningmastery.com/blog/feed/",
    "https://thegradient.pub/feed/",
    "https://openai.com/news/rss.xml",
    "https://www.aitrends.com/feed/"

]

for url in RSS_FEEDS:
    feed = feedparser.parse(url)
    print(f"Flux : {url} → {len(feed.entries)} articles disponibles")

SUMMARY_SENTENCES = 5  # nombre de phrases pour le résumé

# ==============================
# 2. Chargement spaCy
# ==============================
nlp = spacy.load("en_core_web_sm")

# ==============================
# 3. Namespaces RDF
# ==============================
EX = Namespace("http://example.org/")
SCHEMA = Namespace("http://schema.org/")

# ==============================
# 4. Graphe RDF global
# ==============================
global_graph = Graph()
global_graph.bind("ex", EX)
global_graph.bind("schema", SCHEMA)
global_graph.bind("foaf", FOAF)

# ==============================
# 5. Anthologie globale
# ==============================
global_keywords = {}

# ==============================
# 6. Traitement de chaque flux RSS
# ==============================
for feed_url in RSS_FEEDS:
    feed = feedparser.parse(feed_url)
    source_name = feed.feed.get("title", "Unknown Source")
    source_url = feed.feed.get("link", feed_url)
    source_hash = hashlib.md5(source_name.encode("utf-8")).hexdigest()
    source_uri = EX[f"source/{source_hash}"]

    print(f"\n=== Traitement du flux : {source_name} ({len(feed.entries)} articles) ===")

    # Ajouter la source dans le graphe global
    global_graph.add((source_uri, RDF.type, SCHEMA.Organization))
    global_graph.add((source_uri, SCHEMA.name, Literal(source_name, datatype=XSD.string)))
    global_graph.add((source_uri, SCHEMA.url, Literal(source_url, datatype=XSD.anyURI)))

    # Parcours des articles
    for idx, entry in enumerate(feed.entries, start=1):
        title = entry.get("title", "").strip()
        summary = entry.get("summary", "").strip()
        content = ""
        if "content" in entry:
            content = " ".join(c.get("value", "") for c in entry.content)

        article_text = "\n".join(filter(None, [title, summary, content])).strip()

        # === Debug : vérifier pourquoi certains articles sont vides ===
        print(f"DEBUG: Article {idx} - titre = {title}, longueur texte = {len(article_text)}")

        # Fallback si texte vide
        if not article_text:
            article_text = title

        article_url = entry.get("link", f"rss://article/{idx}")
        article_hash = hashlib.md5(article_url.encode("utf-8")).hexdigest()
        article_uri = EX[f"article/{article_hash}"]

        published = entry.get("published", entry.get("updated", None))
        published_display = published if published else "N/A"

        # ===== Résumé automatique avec Sumy =====
        parser = PlaintextParser.from_string(article_text, Tokenizer("english"))
        summarizer = LexRankSummarizer()
        summary_sentences = summarizer(parser.document, SUMMARY_SENTENCES)
        summary_text = " ".join([str(sentence) for sentence in summary_sentences])

        # ===== Extraction des mots-clés =====
        doc = nlp(article_text)
        keywords = set()

        # Entités nommées
        for ent in doc.ents:
            if ent.label_ in ["ORG", "PERSON", "GPE", "PRODUCT", "EVENT"]:
                keywords.add(ent.text.strip())

        # Noms communs et propres
        for token in doc:
            if token.pos_ in ["NOUN", "PROPN"] and len(token.text) > 2:
                keywords.add(token.text.strip())

        # Mettre à jour l’anthologie globale
        for kw in keywords:
            global_keywords[kw] = global_keywords.get(kw, 0) + 1

        # ===== Impression résumé + mots-clés =====
        print(f"\n--- ARTICLE {idx} ---")
        print(f"Titre : {title}")
        print(f"URL   : {article_url}")
        print(f"Date  : {published_display}")
        print(f"Résumé automatique : {summary_text}")
        print(f"Mots-clés : {', '.join(list(keywords)[:10])}")

        # ===== Création du graphe RDF pour cet article =====
        g = Graph()
        g.bind("ex", EX)
        g.bind("schema", SCHEMA)
        g.bind("foaf", FOAF)

        # Source
        g.add((source_uri, RDF.type, SCHEMA.Organization))
        g.add((source_uri, SCHEMA.name, Literal(source_name, datatype=XSD.string)))
        g.add((source_uri, SCHEMA.url, Literal(source_url, datatype=XSD.anyURI)))

        # Article
        g.add((article_uri, RDF.type, SCHEMA.Article))
        g.add((article_uri, SCHEMA.headline, Literal(title, datatype=XSD.string)))
        g.add((article_uri, SCHEMA.url, Literal(article_url, datatype=XSD.anyURI)))
        g.add((article_uri, SCHEMA.publisher, source_uri))
        if published:
            g.add((article_uri, SCHEMA.datePublished, Literal(published, datatype=XSD.string)))
        g.add((article_uri, SCHEMA.abstract, Literal(summary_text, datatype=XSD.string)))
        g.add((article_uri, SCHEMA.keywords, Literal(", ".join(keywords), datatype=XSD.string)))

        # Organisations extraites
        orgs = [ent.text.strip() for ent in doc.ents if ent.label_ == "ORG"]
        for org in orgs:
            org_hash = hashlib.md5(org.lower().encode("utf-8")).hexdigest()
            org_uri = EX[f"org/{org_hash}"]
            g.add((org_uri, RDF.type, SCHEMA.Organization))
            g.add((org_uri, SCHEMA.name, Literal(org, datatype=XSD.string)))
            g.add((article_uri, SCHEMA.about, org_uri))

        # Ajouter dans le graphe global
        for triple in g:
            global_graph.add(triple)

        # ===== Visualisation du graphe pour cet article =====
        nx_graph = nx.DiGraph()
        for s, p, o in g.triples((None, None, None)):
            nx_graph.add_node(s, label=str(s).split('/')[-1])
            nx_graph.add_node(o, label=str(o).split('/')[-1])
            nx_graph.add_edge(s, o, label=str(p).split('/')[-1])

        pos = nx.spring_layout(nx_graph, k=1.0, iterations=100)  # graphe plus étiré
        plt.figure(figsize=(12, 7))
        nx.draw(
            nx_graph, pos,
            with_labels=True,
            labels=nx.get_node_attributes(nx_graph, 'label'),
            node_size=2500,
            node_color='lightblue',
            font_size=9,
            font_weight='bold',
            arrowsize=20
        )
        edge_labels = nx.get_edge_attributes(nx_graph, 'label')
        nx.draw_networkx_edge_labels(nx_graph, pos, edge_labels=edge_labels, font_size=8)
        plt.title(f"Graphe RDF : {title[:50]}...", fontsize=14)
        plt.axis('off')
        plt.show()
        plt.close()

# ==============================
# 7. Affichage de l’anthologie globale
# ==============================
print("\n=== ANTHOLOGIE GLOBALE DES MOTS-CLÉS ===")
sorted_keywords = sorted(global_keywords.items(), key=lambda x: x[1], reverse=True)
for kw, freq in sorted_keywords[:20]:
    print(f"{kw} ({freq})")

# ==============================
# 8. Export RDF global
# ==============================
global_graph.serialize(destination="rss_articles_knowledge_graph.ttl", format="turtle")
print("\nExport RDF terminé : rss_articles_knowledge_graph.ttl")


ModuleNotFoundError: No module named 'rdflib'

*******************************************************************************************************************************************************************
**FLUX RSS HAJAR **
************************************************************************************************************************************************************************************************

In [ ]:
import feedparser
import spacy
import hashlib
from rdflib import Graph, Namespace, Literal, RDF
from rdflib.namespace import XSD, FOAF
import networkx as nx
import matplotlib.pyplot as plt
import nltk

# ===== Sumy =====
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

# NLTK tokenizer
nltk.download('punkt')

# ==============================
# 1. Configuration
# ==============================

RSS_FEEDS = [
    "https://fintech.global/feed/",
"https://www.ft.com/?format=rss",
"https://www.franceassureurs.fr/actualites/feed/",
"http://www.fnof.org/category/actu/feed/",
"https://www.google.com/alerts/feeds/16286487246621706843/7494204588410233626",
"https://www.irdes.fr/rssEspaceDoc.xml",
"https://www.irdes.fr/rssEspaceEnseignement.xml",
"https://www.irdes.fr/rssEspaceRecherche.xml",
"https://www.insee.fr/fr/flux/1",
"https://insuranceasianews.com/feed/",
"https://www.lassuranceenmouvement.com/feed/",
"http://rss.lsa-conso.fr/Lsa-rss",
"https://www.ladepeche.fr/rss.xml",
"https://www.la-croix.com/RSS/UNIVERS_ALL",
"https://www.midilibre.fr/rss.xml",
"https://www.mieuxvivre-votreargent.fr/accueil-2/feed/",
"https://www.economie.gouv.fr/rss/toutesactualites",
"https://www.ecologique-solidaire.gouv.fr/rss_actualites.xml",
"https://www.ecologique-solidaire.gouv.fr/rss_presse.xml",
"https://www.mutuelle-land.com/rss/actumutuelleland.xml",
"https://www.newsassurancespro.com/feed",
"https://www.newsassurancespro.com/feed",
"https://www.newsassurancespro.com/comments/feed",
"https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml",
"https://point-banque.fr/feed/",
"https://www.siliconvalley.com/feed/",
"http://www.siliconbeat.com/feed/",
"https://www.sudouest.fr/essentiel/rss.xml",
"https://www.sudouest.fr/essentiel/rss.xml",
"https://technode.com/feed/",
"https://www.theguardian.com/uk/rss",
"https://timesofindia.indiatimes.com/rssfeeds/1898055.cms",
"https://www.uktech.news/feed",
"https://www.sdaudio.org/feed/",
"https://newsfeed.zeit.de/index",
"https://www.sudouest.fr/rss.xml",
"https://www.sudouest.fr/rss.xml",
"https://www.lopinion.fr/rss.xml"
]

for url in RSS_FEEDS:
    feed = feedparser.parse(url)
    print(f"Flux : {url} → {len(feed.entries)} articles disponibles")

SUMMARY_SENTENCES = 5  # nombre de phrases pour le résumé

# ==============================
# 2. Chargement spaCy
# ==============================
nlp = spacy.load("en_core_web_sm")

# ==============================
# 3. Namespaces RDF
# ==============================
EX = Namespace("http://example.org/")
SCHEMA = Namespace("http://schema.org/")

# ==============================
# 4. Graphe RDF global
# ==============================
global_graph = Graph()
global_graph.bind("ex", EX)
global_graph.bind("schema", SCHEMA)
global_graph.bind("foaf", FOAF)

# ==============================
# 5. Anthologie globale
# ==============================
global_keywords = {}

# ==============================
# 6. Traitement de chaque flux RSS
# ==============================
for feed_url in RSS_FEEDS:
    feed = feedparser.parse(feed_url)
    source_name = feed.feed.get("title", "Unknown Source")
    source_url = feed.feed.get("link", feed_url)
    source_hash = hashlib.md5(source_name.encode("utf-8")).hexdigest()
    source_uri = EX[f"source/{source_hash}"]

    print(f"\n=== Traitement du flux : {source_name} ({len(feed.entries)} articles) ===")

    # Ajouter la source dans le graphe global
    global_graph.add((source_uri, RDF.type, SCHEMA.Organization))
    global_graph.add((source_uri, SCHEMA.name, Literal(source_name, datatype=XSD.string)))
    global_graph.add((source_uri, SCHEMA.url, Literal(source_url, datatype=XSD.anyURI)))

    # Parcours des articles
    for idx, entry in enumerate(feed.entries, start=1):
        title = entry.get("title", "").strip()
        summary = entry.get("summary", "").strip()
        content = ""
        if "content" in entry:
            content = " ".join(c.get("value", "") for c in entry.content)

        article_text = "\n".join(filter(None, [title, summary, content])).strip()

        # === Debug : vérifier pourquoi certains articles sont vides ===
        print(f"DEBUG: Article {idx} - titre = {title}, longueur texte = {len(article_text)}")

        # Fallback si texte vide
        if not article_text:
            article_text = title

        article_url = entry.get("link", f"rss://article/{idx}")
        article_hash = hashlib.md5(article_url.encode("utf-8")).hexdigest()
        article_uri = EX[f"article/{article_hash}"]

        published = entry.get("published", entry.get("updated", None))
        published_display = published if published else "N/A"

        # ===== Résumé automatique avec Sumy =====
        parser = PlaintextParser.from_string(article_text, Tokenizer("english"))
        summarizer = LexRankSummarizer()
        summary_sentences = summarizer(parser.document, SUMMARY_SENTENCES)
        summary_text = " ".join([str(sentence) for sentence in summary_sentences])

        # ===== Extraction des mots-clés =====
        doc = nlp(article_text)
        keywords = set()

        # Entités nommées
        for ent in doc.ents:
            if ent.label_ in ["ORG", "PERSON", "GPE", "PRODUCT", "EVENT"]:
                keywords.add(ent.text.strip())

        # Noms communs et propres
        for token in doc:
            if token.pos_ in ["NOUN", "PROPN"] and len(token.text) > 2:
                keywords.add(token.text.strip())

        # Mettre à jour l’anthologie globale
        for kw in keywords:
            global_keywords[kw] = global_keywords.get(kw, 0) + 1

        # ===== Impression résumé + mots-clés =====
        print(f"\n--- ARTICLE {idx} ---")
        print(f"Titre : {title}")
        print(f"URL   : {article_url}")
        print(f"Date  : {published_display}")
        print(f"Résumé automatique : {summary_text}")
        print(f"Mots-clés : {', '.join(list(keywords)[:10])}")

        # ===== Création du graphe RDF pour cet article =====
        g = Graph()
        g.bind("ex", EX)
        g.bind("schema", SCHEMA)
        g.bind("foaf", FOAF)

        # Source
        g.add((source_uri, RDF.type, SCHEMA.Organization))
        g.add((source_uri, SCHEMA.name, Literal(source_name, datatype=XSD.string)))
        g.add((source_uri, SCHEMA.url, Literal(source_url, datatype=XSD.anyURI)))

        # Article
        g.add((article_uri, RDF.type, SCHEMA.Article))
        g.add((article_uri, SCHEMA.headline, Literal(title, datatype=XSD.string)))
        g.add((article_uri, SCHEMA.url, Literal(article_url, datatype=XSD.anyURI)))
        g.add((article_uri, SCHEMA.publisher, source_uri))
        if published:
            g.add((article_uri, SCHEMA.datePublished, Literal(published, datatype=XSD.string)))
        g.add((article_uri, SCHEMA.abstract, Literal(summary_text, datatype=XSD.string)))
        g.add((article_uri, SCHEMA.keywords, Literal(", ".join(keywords), datatype=XSD.string)))

        # Organisations extraites
        orgs = [ent.text.strip() for ent in doc.ents if ent.label_ == "ORG"]
        for org in orgs:
            org_hash = hashlib.md5(org.lower().encode("utf-8")).hexdigest()
            org_uri = EX[f"org/{org_hash}"]
            g.add((org_uri, RDF.type, SCHEMA.Organization))
            g.add((org_uri, SCHEMA.name, Literal(org, datatype=XSD.string)))
            g.add((article_uri, SCHEMA.about, org_uri))

        # Ajouter dans le graphe global
        for triple in g:
            global_graph.add(triple)

        # ===== Visualisation du graphe pour cet article =====
        nx_graph = nx.DiGraph()
        for s, p, o in g.triples((None, None, None)):
            nx_graph.add_node(s, label=str(s).split('/')[-1])
            nx_graph.add_node(o, label=str(o).split('/')[-1])
            nx_graph.add_edge(s, o, label=str(p).split('/')[-1])

        pos = nx.spring_layout(nx_graph, k=1.0, iterations=100)  # graphe plus étiré
        plt.figure(figsize=(12, 7))
        nx.draw(
            nx_graph, pos,
            with_labels=True,
            labels=nx.get_node_attributes(nx_graph, 'label'),
            node_size=2500,
            node_color='lightblue',
            font_size=9,
            font_weight='bold',
            arrowsize=20
        )
        edge_labels = nx.get_edge_attributes(nx_graph, 'label')
        nx.draw_networkx_edge_labels(nx_graph, pos, edge_labels=edge_labels, font_size=8)
        plt.title(f"Graphe RDF : {title[:50]}...", fontsize=14)
        plt.axis('off')
        plt.show()
        plt.close()

# ==============================
# 7. Affichage de l’anthologie globale
# ==============================
print("\n=== ANTHOLOGIE GLOBALE DES MOTS-CLÉS ===")
sorted_keywords = sorted(global_keywords.items(), key=lambda x: x[1], reverse=True)
for kw, freq in sorted_keywords[:20]:
    print(f"{kw} ({freq})")

# ==============================
# 8. Export RDF global
# ==============================
global_graph.serialize(destination="rss_articles_knowledge_graph.ttl", format="turtle")
print("\nExport RDF terminé : rss_articles_knowledge_graph.ttl")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.0 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=7708bb02678a38f138f24e350077cc7beca3ca7db200b0202a804558a1120dcb
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


ModuleNotFoundError: No module named 'rdflib'

IMPORT FICHIER

In [ ]:
# =========================================================
# INSTALLS (à lancer une fois)
# =========================================================
!pip install feedparser spacy sumy nltk beautifulsoup4 lxml

# =========================================================
# IMPORTS
# =========================================================
import feedparser
import spacy
import nltk
import html
import re

from bs4 import BeautifulSoup
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

nltk.download("punkt")

# =========================================================
# FONCTION NETTOYAGE HTML
# =========================================================
def clean_html(text: str) -> str:
    if not text:
        return ""

    # Décodage entités HTML
    text = html.unescape(text)

    # Suppression balises HTML
    soup = BeautifulSoup(text, "lxml")
    text = soup.get_text(separator=" ")

    # Nettoyage espaces
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# =========================================================
# CONFIG
# =========================================================
RSS_FEEDS = [
    "https://fintech.global/feed/",
    "https://www.ft.com/?format=rss",
    "https://www.franceassureurs.fr/actualites/feed/",
    "http://www.fnof.org/category/actu/feed/",
    "https://www.google.com/alerts/feeds/16286487246621706843/7494204588410233626",
    "https://www.irdes.fr/rssEspaceDoc.xml",
    "https://www.irdes.fr/rssEspaceEnseignement.xml",
    "https://www.irdes.fr/rssEspaceRecherche.xml",
    "https://www.insee.fr/fr/flux/1",
    "https://insuranceasianews.com/feed/",
    "https://www.lassuranceenmouvement.com/feed/",
    "http://rss.lsa-conso.fr/Lsa-rss",
    "https://www.ladepeche.fr/rss.xml",
    "https://www.la-croix.com/RSS/UNIVERS_ALL",
    "https://www.midilibre.fr/rss.xml",
    "https://www.mieuxvivre-votreargent.fr/accueil-2/feed/",
    "https://www.economie.gouv.fr/rss/toutesactualites",
    "https://www.ecologique-solidaire.gouv.fr/rss_actualites.xml",
    "https://www.ecologique-solidaire.gouv.fr/rss_presse.xml",
    "https://www.mutuelle-land.com/rss/actumutuelleland.xml",
    "https://www.newsassurancespro.com/feed",
    "https://www.newsassurancespro.com/comments/feed",
    "https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml",
    "https://point-banque.fr/feed/",
    "https://www.siliconvalley.com/feed/",
    "http://www.siliconbeat.com/feed/",
    "https://www.sudouest.fr/essentiel/rss.xml",
    "https://technode.com/feed/",
    "https://www.theguardian.com/uk/rss",
    "https://timesofindia.indiatimes.com/rssfeeds/1898055.cms",
    "https://www.uktech.news/feed",
    "https://www.sdaudio.org/feed/",
    "https://newsfeed.zeit.de/index",
    "https://www.sudouest.fr/rss.xml",
    "https://www.lopinion.fr/rss.xml"
]

SUMMARY_SENTENCES = 5
OUTPUT_FILE = "rss_export_structure.txt"

# =========================================================
# spaCy
# =========================================================
nlp = spacy.load("en_core_web_sm")

# =========================================================
# STOCKAGE
# =========================================================
flux_info = []
articles_by_flux = {}

# =========================================================
# TRAITEMENT
# =========================================================
for flux_index, feed_url in enumerate(RSS_FEEDS, start=1):
    feed = feedparser.parse(feed_url)

    flux_name = feed.feed.get("title", "Flux inconnu")
    flux_link = feed.feed.get("link", feed_url)
    entries = feed.entries

    flux_info.append({
        "index": flux_index,
        "name": flux_name,
        "url": flux_link,
        "count": len(entries)
    })

    articles_by_flux[flux_name] = []

    for idx, entry in enumerate(entries, start=1):
        title = clean_html(entry.get("title", "").strip())

        raw_summary = entry.get("summary", "")
        raw_content = ""

        if "content" in entry:
            raw_content = " ".join(c.get("value", "") for c in entry.content)

        summary = clean_html(raw_summary)
        content = clean_html(raw_content)

        text = "\n".join(filter(None, [title, summary, content]))
        if not text:
            text = title

        published = entry.get("published", entry.get("updated", "N/A"))
        url = entry.get("link", "N/A")

        # ===== Résumé automatique =====
        parser = PlaintextParser.from_string(text, Tokenizer("english"))
        summarizer = LexRankSummarizer()
        summary_text = " ".join(
            str(s) for s in summarizer(parser.document, SUMMARY_SENTENCES)
        )

        # ===== Mots-clés =====
        doc = nlp(text)
        keywords = set()

        for ent in doc.ents:
            if ent.label_ in ["ORG", "PERSON", "GPE", "PRODUCT", "EVENT"]:
                keywords.add(ent.text.strip())

        for token in doc:
            if token.pos_ in ["NOUN", "PROPN"] and len(token.text) > 2:
                keywords.add(token.text.strip())

        articles_by_flux[flux_name].append({
            "index": idx,
            "date": published,
            "url": url,
            "summary": summary_text,
            "keywords": ", ".join(sorted(keywords))
        })

# =========================================================
# ÉCRITURE DU FICHIER
# =========================================================
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    f.write("=" * 50 + "\n")
    f.write("RÉSUMÉ DES FLUX RSS\n")
    f.write("=" * 50 + "\n\n")

    for flux in flux_info:
        f.write(f"[{flux['index']}] {flux['name']}\n")
        f.write(f"URL : {flux['url']}\n")
        f.write(f"Nombre d’articles lus : {flux['count']}\n\n")

    f.write("\n" + "=" * 50 + "\n")
    f.write("DÉTAIL DES ARTICLES\n")
    f.write("=" * 50 + "\n\n")

    for flux_name, articles in articles_by_flux.items():
        f.write("-" * 50 + "\n")
        f.write(f"Flux : {flux_name}\n")
        f.write("-" * 50 + "\n\n")

        for article in articles:
            f.write(f"Article #{article['index']}\n")
            f.write(f"Date  : {article['date']}\n")
            f.write(f"URL   : {article['url']}\n\n")

            f.write("Résumé :\n")
            f.write(article["summary"] + "\n\n")

            f.write("Mots-clés :\n")
            f.write(article["keywords"] + "\n")
            f.write("\n" + "-" * 50 + "\n\n")

print(f"✅ Fichier généré proprement : {OUTPUT_FILE}")


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 67.9 MB/s eta 0:00:00
  Created wheel for breadability: filename=breadability-0.1.20-py2.py3-none-any.whl size=21695 sha256=5928a783e63ae5303887371bd711ef9c6d875f31e47633ce41b1167f520651b9
  Stored in directory: /root/.cache/pip/wheels/32/99/64/59305409cacd03aa03e7bddf31a9db34b1fa7033bd41972662
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=d0fd61723e353ff6c96bfca78bb24a0640f82a6f1fc9dc5964f0b944ffe57b07
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=4eae052148406f27a74

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


LookupError: NLTK tokenizers are missing or the language is not supported.
Download them by following command: python -c "import nltk; nltk.download('punkt')"
Original error was:

**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


**PROBLEME XML A RESOUDRE **

In [ ]:
import feedparser
import spacy
import nltk

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

nltk.download("punkt")

# ==============================
# CONFIG
# ==============================

RSS_FEEDS = [
    "https://fintech.global/feed/",
"https://www.ft.com/?format=rss",
"https://www.franceassureurs.fr/actualites/feed/",
"http://www.fnof.org/category/actu/feed/",
"https://www.google.com/alerts/feeds/16286487246621706843/7494204588410233626",
"https://www.irdes.fr/rssEspaceDoc.xml",
"https://www.irdes.fr/rssEspaceEnseignement.xml",
"https://www.irdes.fr/rssEspaceRecherche.xml",
"https://www.insee.fr/fr/flux/1",
"https://insuranceasianews.com/feed/",
"https://www.lassuranceenmouvement.com/feed/",
"http://rss.lsa-conso.fr/Lsa-rss",
"https://www.ladepeche.fr/rss.xml",
"https://www.la-croix.com/RSS/UNIVERS_ALL",
"https://www.midilibre.fr/rss.xml",
"https://www.mieuxvivre-votreargent.fr/accueil-2/feed/",
"https://www.economie.gouv.fr/rss/toutesactualites",
"https://www.ecologique-solidaire.gouv.fr/rss_actualites.xml",
"https://www.ecologique-solidaire.gouv.fr/rss_presse.xml",
"https://www.mutuelle-land.com/rss/actumutuelleland.xml",
"https://www.newsassurancespro.com/feed",
"https://www.newsassurancespro.com/feed",
"https://www.newsassurancespro.com/comments/feed",
"https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml",
"https://point-banque.fr/feed/",
"https://www.siliconvalley.com/feed/",
"http://www.siliconbeat.com/feed/",
"https://www.sudouest.fr/essentiel/rss.xml",
"https://www.sudouest.fr/essentiel/rss.xml",
"https://technode.com/feed/",
"https://www.theguardian.com/uk/rss",
"https://timesofindia.indiatimes.com/rssfeeds/1898055.cms",
"https://www.uktech.news/feed",
"https://www.sdaudio.org/feed/",
"https://newsfeed.zeit.de/index",
"https://www.sudouest.fr/rss.xml",
"https://www.sudouest.fr/rss.xml",
"https://www.lopinion.fr/rss.xml"

]

SUMMARY_SENTENCES = 5
OUTPUT_FILE = "test1.txt"

nlp = spacy.load("en_core_web_sm")

# ==============================
# STOCKAGE
# ==============================

flux_info = []
articles_by_flux = {}

# ==============================
# TRAITEMENT
# ==============================

for flux_index, feed_url in enumerate(RSS_FEEDS, start=1):
    feed = feedparser.parse(feed_url)

    flux_name = feed.feed.get("title", "Flux inconnu")
    flux_link = feed.feed.get("link", feed_url)
    entries = feed.entries

    flux_info.append({
        "index": flux_index,
        "name": flux_name,
        "url": flux_link,
        "count": len(entries)
    })

    articles_by_flux[flux_name] = []

    for idx, entry in enumerate(entries, start=1):
        title = entry.get("title", "").strip()
        summary = entry.get("summary", "").strip()
        content = ""

        if "content" in entry:
            content = " ".join(c.get("value", "") for c in entry.content)

        text = "\n".join(filter(None, [title, summary, content]))
        if not text:
            text = title

        published = entry.get("published", entry.get("updated", "N/A"))
        url = entry.get("link", "N/A")

        # ===== Résumé =====
        parser = PlaintextParser.from_string(text, Tokenizer("english"))
        summarizer = LexRankSummarizer()
        summary_text = " ".join(str(s) for s in summarizer(parser.document, SUMMARY_SENTENCES))

        # ===== Mots-clés =====
        doc = nlp(text)
        keywords = set()

        for ent in doc.ents:
            if ent.label_ in ["ORG", "PERSON", "GPE", "PRODUCT", "EVENT"]:
                keywords.add(ent.text)

        for token in doc:
            if token.pos_ in ["NOUN", "PROPN"] and len(token.text) > 2:
                keywords.add(token.text)

        articles_by_flux[flux_name].append({
            "index": idx,
            "date": published,
            "url": url,
            "summary": summary_text,
            "keywords": ", ".join(sorted(keywords))
        })

# ==============================
# ÉCRITURE DU FICHIER
# ==============================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    f.write("=" * 50 + "\n")
    f.write("RÉSUMÉ DES FLUX RSS\n")
    f.write("=" * 50 + "\n\n")

    for flux in flux_info:
        f.write(f"[{flux['index']}] {flux['name']}\n")
        f.write(f"URL : {flux['url']}\n")
        f.write(f"Nombre d’articles lus : {flux['count']}\n\n")

    f.write("\n" + "=" * 50 + "\n")
    f.write("DÉTAIL DES ARTICLES\n")
    f.write("=" * 50 + "\n\n")

    for flux_name, articles in articles_by_flux.items():
        f.write("-" * 50 + "\n")
        f.write(f"Flux : {flux_name}\n")
        f.write("-" * 50 + "\n\n")

        for article in articles:
            f.write(f"Article #{article['index']}\n")
            f.write(f"Date  : {article['date']}\n")
            f.write(f"URL   : {article['url']}\n")
            f.write("Résumé :\n")
            f.write(article["summary"] + "\n\n")
            f.write("Mots-clés :\n")
            f.write(article["keywords"] + "\n")
            f.write("\n" + "-" * 50 + "\n\n")

print(f"✅ Fichier généré : {OUTPUT_FILE}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


✅ Fichier généré : test1.txt


******************************************************************************************************************************************************
**V2 modifs HTML**
**********************************************************************************************************************************************************


In [ ]:
# =========================================================
# IMPORTS
# =========================================================
import feedparser
import spacy
import nltk
import html
import re

from bs4 import BeautifulSoup
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

nltk.download("punkt")

# =========================================================
# FONCTION NETTOYAGE HTML (ROBUSTE)
# =========================================================
def clean_html(text: str) -> str:
    if not text:
        return ""

    # Décoder entités HTML
    text = html.unescape(text)

    soup = BeautifulSoup(text, "lxml")

    # Supprimer balises inutiles
    for tag in soup(["script", "style", "figure", "figcaption"]):
        tag.decompose()

    # Supprimer liens parasites (Guardian, FT, etc.)
    for a in soup.find_all("a"):
        if "continue reading" in a.get_text().lower():
            a.decompose()

    # Extraire texte
    text = soup.get_text(separator=" ")

    # Nettoyage final
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# =========================================================
# CONFIG
# =========================================================
RSS_FEEDS = [
    "https://fintech.global/feed/",
    "https://www.ft.com/?format=rss",
    "https://www.franceassureurs.fr/actualites/feed/",
    "http://www.fnof.org/category/actu/feed/",
    "https://www.google.com/alerts/feeds/16286487246621706843/7494204588410233626",
    "https://www.irdes.fr/rssEspaceDoc.xml",
    "https://www.irdes.fr/rssEspaceEnseignement.xml",
    "https://www.irdes.fr/rssEspaceRecherche.xml",
    "https://www.insee.fr/fr/flux/1",
    "https://insuranceasianews.com/feed/",
    "https://www.lassuranceenmouvement.com/feed/",
    "http://rss.lsa-conso.fr/Lsa-rss",
    "https://www.ladepeche.fr/rss.xml",
    "https://www.la-croix.com/RSS/UNIVERS_ALL",
    "https://www.midilibre.fr/rss.xml",
    "https://www.mieuxvivre-votreargent.fr/accueil-2/feed/",
    "https://www.economie.gouv.fr/rss/toutesactualites",
    "https://www.ecologique-solidaire.gouv.fr/rss_actualites.xml",
    "https://www.ecologique-solidaire.gouv.fr/rss_presse.xml",
    "https://www.mutuelle-land.com/rss/actumutuelleland.xml",
    "https://www.newsassurancespro.com/feed",
    "https://www.newsassurancespro.com/comments/feed",
    "https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml",
    "https://point-banque.fr/feed/",
    "https://www.siliconvalley.com/feed/",
    "http://www.siliconbeat.com/feed/",
    "https://www.sudouest.fr/essentiel/rss.xml",
    "https://technode.com/feed/",
    "https://www.theguardian.com/uk/rss",
    "https://timesofindia.indiatimes.com/rssfeeds/1898055.cms",
    "https://www.uktech.news/feed",
    "https://www.sdaudio.org/feed/",
    "https://newsfeed.zeit.de/index",
    "https://www.sudouest.fr/rss.xml",
    "https://www.lopinion.fr/rss.xml"
]

SUMMARY_SENTENCES = 5
OUTPUT_FILE = "test1.txt"
MAX_TEXT_LENGTH = 3000   # 🔥 anti Guardian / FT trop longs

# =========================================================
# spaCy
# =========================================================
nlp = spacy.load("en_core_web_sm")

# =========================================================
# STOCKAGE
# =========================================================
flux_info = []
articles_by_flux = {}

# =========================================================
# TRAITEMENT
# =========================================================
for flux_index, feed_url in enumerate(RSS_FEEDS, start=1):
    feed = feedparser.parse(feed_url)

    flux_name = feed.feed.get("title", "Flux inconnu")
    flux_link = feed.feed.get("link", feed_url)
    entries = feed.entries

    flux_info.append({
        "index": flux_index,
        "name": flux_name,
        "url": flux_link,
        "count": len(entries)
    })

    articles_by_flux[flux_name] = []

    for idx, entry in enumerate(entries, start=1):

        # ===== TITRE =====
        title = clean_html(entry.get("title", ""))

        # ===== SUMMARY (HTML SAFE) =====
        if "summary_detail" in entry and entry.summary_detail.get("type") == "text/html":
            raw_summary = entry.summary_detail.get("value", "")
        else:
            raw_summary = entry.get("summary", "")

        summary = clean_html(raw_summary)

        # ===== CONTENT =====
        raw_content = ""
        if "content" in entry:
            raw_content = " ".join(c.get("value", "") for c in entry.content)

        content = clean_html(raw_content)

        # ===== TEXTE FINAL =====
        text = "\n".join(filter(None, [title, summary, content]))
        text = text[:MAX_TEXT_LENGTH]

        if not text:
            text = title

        published = entry.get("published", entry.get("updated", "N/A"))
        url = entry.get("link", "N/A")

        # ===== RÉSUMÉ AUTOMATIQUE =====
        parser = PlaintextParser.from_string(text, Tokenizer("english"))
        summarizer = LexRankSummarizer()
        summary_text = " ".join(
            str(s) for s in summarizer(parser.document, SUMMARY_SENTENCES)
        )

        # ===== MOTS-CLÉS =====
        doc = nlp(text)
        keywords = set()

        for ent in doc.ents:
            if ent.label_ in ["ORG", "PERSON", "GPE", "PRODUCT", "EVENT"]:
                keywords.add(ent.text.strip())

        for token in doc:
            if token.pos_ in ["NOUN", "PROPN"] and len(token.text) > 2:
                keywords.add(token.text.strip())

        articles_by_flux[flux_name].append({
            "index": idx,
            "date": published,
            "url": url,
            "summary": summary_text,
            "keywords": ", ".join(sorted(keywords))
        })

# =========================================================
# ÉCRITURE DU FICHIER
# =========================================================
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    f.write("=" * 50 + "\n")
    f.write("RÉSUMÉ DES FLUX RSS\n")
    f.write("=" * 50 + "\n\n")

    for flux in flux_info:
        f.write(f"[{flux['index']}] {flux['name']}\n")
        f.write(f"URL : {flux['url']}\n")
        f.write(f"Nombre d’articles lus : {flux['count']}\n\n")

    f.write("\n" + "=" * 50 + "\n")
    f.write("DÉTAIL DES ARTICLES\n")
    f.write("=" * 50 + "\n\n")

    for flux_name, articles in articles_by_flux.items():
        f.write("-" * 50 + "\n")
        f.write(f"Flux : {flux_name}\n")
        f.write("-" * 50 + "\n\n")

        for article in articles:
            f.write(f"Article #{article['index']}\n")
            f.write(f"Date  : {article['date']}\n")
            f.write(f"URL   : {article['url']}\n\n")

            f.write("Résumé :\n")
            f.write(article["summary"] + "\n\n")

            f.write("Mots-clés :\n")
            f.write(article["keywords"] + "\n")
            f.write("\n" + "-" * 50 + "\n\n")

print(f"✅ Fichier généré proprement : {OUTPUT_FILE}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


✅ Fichier généré proprement : test1.txt


HTML  DANS MOTS CLE A RETIRER

In [ ]:
import feedparser
import spacy
import nltk
import re
import html

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

nltk.download("punkt")

# ==============================
# CONFIG
# ==============================

RSS_FEEDS = [
    "https://fintech.global/feed/",
    "https://www.ft.com/?format=rss",
    "https://www.franceassureurs.fr/actualites/feed/",
    "http://www.fnof.org/category/actu/feed/",
    "https://www.google.com/alerts/feeds/16286487246621706843/7494204588410233626",
    "https://www.irdes.fr/rssEspaceDoc.xml",
    "https://www.irdes.fr/rssEspaceEnseignement.xml",
    "https://www.irdes.fr/rssEspaceRecherche.xml",
    "https://www.insee.fr/fr/flux/1",
    "https://insuranceasianews.com/feed/",
    "https://www.lassuranceenmouvement.com/feed/",
    "http://rss.lsa-conso.fr/Lsa-rss",
    "https://www.ladepeche.fr/rss.xml",
    "https://www.la-croix.com/RSS/UNIVERS_ALL",
    "https://www.midilibre.fr/rss.xml",
    "https://www.mieuxvivre-votreargent.fr/accueil-2/feed/",
    "https://www.economie.gouv.fr/rss/toutesactualites",
    "https://www.ecologique-solidaire.gouv.fr/rss_actualites.xml",
    "https://www.ecologique-solidaire.gouv.fr/rss_presse.xml",
    "https://www.mutuelle-land.com/rss/actumutuelleland.xml",
    "https://www.newsassurancespro.com/feed",
    "https://www.newsassurancespro.com/comments/feed",
    "https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml",
    "https://point-banque.fr/feed/",
    "https://www.siliconvalley.com/feed/",
    "https://technode.com/feed/",
    "https://www.theguardian.com/uk/rss",
    "https://www.uktech.news/feed",
    "https://newsfeed.zeit.de/index",
    "https://www.lopinion.fr/rss.xml"
]

SUMMARY_SENTENCES = 5
OUTPUT_FILE = "test2.txt"

nlp = spacy.load("en_core_web_sm")

# ==============================
# NETTOYAGE TEXTE
# ==============================

def clean_html(text: str) -> str:
    if not text:
        return ""
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)      # balises HTML
    text = re.sub(r"http\S+", " ", text)      # URLs
    text = re.sub(r"\s+", " ", text)          # espaces multiples
    return text.strip()

def is_valid_keyword(token: str) -> bool:
    token = token.strip().lower()

    if len(token) < 3:
        return False

    if any(char.isdigit() for char in token):
        return False

    if any(x in token for x in [
        "/", "\\", "=", "?", "&", "%", "_", "-", ".", ":"
    ]):
        return False

    blacklist = {
        "http", "https", "www", "rss", "image", "video",
        "width", "height", "src", "href", "uploads",
        "content", "date", "title", "time", "file"
    }

    if token in blacklist:
        return False

    return True

# ==============================
# STOCKAGE
# ==============================

flux_info = []
articles_by_flux = {}

# ==============================
# TRAITEMENT
# ==============================

for flux_index, feed_url in enumerate(RSS_FEEDS, start=1):
    feed = feedparser.parse(feed_url)

    flux_name = feed.feed.get("title", "Flux inconnu")
    flux_link = feed.feed.get("link", feed_url)
    entries = feed.entries

    flux_info.append({
        "index": flux_index,
        "name": flux_name,
        "url": flux_link,
        "count": len(entries)
    })

    articles_by_flux[flux_name] = []

    for idx, entry in enumerate(entries, start=1):
        title = clean_html(entry.get("title", ""))
        summary = clean_html(entry.get("summary", ""))

        content = ""
        if "content" in entry:
            content = " ".join(
                clean_html(c.get("value", "")) for c in entry.content
            )

        text = "\n".join(filter(None, [title, summary, content]))
        if not text:
            text = title

        published = entry.get("published", entry.get("updated", "N/A"))
        url = entry.get("link", "N/A")

        # ===== RÉSUMÉ =====
        parser = PlaintextParser.from_string(text, Tokenizer("english"))
        summarizer = LexRankSummarizer()
        summary_text = " ".join(
            str(s) for s in summarizer(parser.document, SUMMARY_SENTENCES)
        )

        # ===== MOTS-CLÉS (NETTOYÉS) =====
        doc = nlp(text)
        keywords = set()

        for ent in doc.ents:
            if ent.label_ in ["ORG", "PERSON", "GPE", "PRODUCT", "EVENT"]:
                if is_valid_keyword(ent.text):
                    keywords.add(ent.text)

        for token in doc:
            if token.pos_ in ["NOUN", "PROPN"]:
                if is_valid_keyword(token.text):
                    keywords.add(token.text)

        articles_by_flux[flux_name].append({
            "index": idx,
            "date": published,
            "url": url,
            "summary": summary_text,
            "keywords": ", ".join(sorted(keywords))
        })

# ==============================
# ÉCRITURE FICHIER
# ==============================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    f.write("=" * 60 + "\n")
    f.write("RÉSUMÉ DES FLUX RSS\n")
    f.write("=" * 60 + "\n\n")

    for flux in flux_info:
        f.write(f"[{flux['index']}] {flux['name']}\n")
        f.write(f"URL : {flux['url']}\n")
        f.write(f"Nombre d’articles lus : {flux['count']}\n\n")

    f.write("\n" + "=" * 60 + "\n")
    f.write("DÉTAIL DES ARTICLES\n")
    f.write("=" * 60 + "\n\n")

    for flux_name, articles in articles_by_flux.items():
        f.write("-" * 60 + "\n")
        f.write(f"Flux : {flux_name}\n")
        f.write("-" * 60 + "\n\n")

        for article in articles:
            f.write(f"Article #{article['index']}\n")
            f.write(f"Date  : {article['date']}\n")
            f.write(f"URL   : {article['url']}\n\n")
            f.write("Résumé :\n")
            f.write(article["summary"] + "\n\n")
            f.write("Mots-clés :\n")
            f.write(article["keywords"] + "\n")
            f.write("\n" + "-" * 60 + "\n\n")

print(f"✅ Fichier généré : {OUTPUT_FILE}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


✅ Fichier généré : test2.txt
